# FULL ADJUSTABLE TRANSIT TOOL

### (fill in necessary links/values and run all)

In [ ]:
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
from folium.plugins import HeatMap
import xgboost as xgb
import matplotlib.pyplot as plt
import warnings
import sys
import math
from sklearn.linear_model import LinearRegression
sys.path.append('../..')
#from config import CENSUS_KEY
warnings.filterwarnings('ignore')
MODEL_PATH  = '../../models/xgb_model_2.json' 
CENSUS_KEY = ''
CENSUS_VARIABLES = [
    'B01003_001E',  # total population
    'B08201_001E',  # total households
    'B08201_002E',  # zero-vehicle households
    'B08201_003E',  # one-vehicle households
    'B08301_001E',  # total commuters
    'B08301_002E',  # drove alone
    'B08301_004E',  # carpooled
    'B08301_010E',  # public transit
    'B23025_001E',  # population 16+ (labor force universe)
    'B23025_004E',  # employed civilians
    'B25003_001E',  # total occupied housing units
    'B25003_003E',  # renter-occupied
    'B19013_001E',  # median household income
]

COUNTIES = [
    #Add county FIPS codes here (last 3 digits of the 5-digit code)
    # '001',  # Adams County
    # '005',  # Arapahoe County
    # # '013',  # Boulder County
    # # '014',  # Broomfield County
    # '059',  # Jefferson County
    # '031',  # Denver County

    # '001',  # Alameda
    # '075',  # San Francisco
    # '081',  # San Mateo
    # '085',  # Santa Clara
    # '013',  # Contra Costa
]
GRID_SPACING_KM  = 0.5       # distance between candidate grid points
N_STATIONS       = 15        # station budget
MIN_SPACING_KM   = 3.0       # minimum distance between any two selected stations

MIN_JOB_KM = 2.0 #minimum distance between job centers

# BOUND BOX AROUND AREA OF INTEREST (min_lon, min_lat, max_lon, max_lat)
BOUNDS = ()

# BOUNDS = (-123.55, 36.85, -121.20, 38.85) 
# for Silicon Valley
# BOUNDS = (-122.75, 37.15, -121.55, 38.05) 
# for San Francisco Bay Area
# BOUNDS = (-105.35, 39.55, -104.60, 40.25) 
# for Denver Metro Area
#BOUNDS = (-119.240929, 33.355132, -117.794815, 34.448494) 
# for Los Angeles Metro Area (LA, Orange, Ventura counties

BOUND_AVG = ((BOUNDS[0] + BOUNDS[2]) / 2, (BOUNDS[1] + BOUNDS[3]) / 2) # average lat/lon for centering maps


STATE_CODE = '06' # 2-digit state FIPS code (06 for California, 08 for Colorado etc)
STATE_ABBV = 'ca' # LOWERCASE state abbreviation (ca for California, co for Colorado etc)

WEEK = '' #1-52 (Defaults to middle of the year if '')

# MAX_LINES = 10 # max number of train lines
# Training features
FEATURE_COLS = [
    'Month',
    'Week',
    'population_density',
    'low_car_rate',
    'transit_commuter_rate',
    'driver_rate',
    'carpool_rate',
    'employment_rate',
    'renter_rate',
    'median_income',
    'is_post_covid',
    'dist_to_nearest_job_center_km'
]

#Census Features
CENSUS_FEATURE_COLS = [
    'population_density',
    'low_car_rate',
    'transit_commuter_rate',
    'driver_rate',
    'carpool_rate',
    'employment_rate',
    'renter_rate',
    'median_income',
]

### Pulling ACS Census Data

In [ ]:
vars_str = ','.join(CENSUS_VARIABLES)
county_str = '&in=county:' + ','.join(COUNTIES)

#pulling census data using requests
url = (
    f'https://api.census.gov/data/2023/acs/acs5'
    f'?get=NAME,{vars_str}'
    f'&for=tract:*'
    f'&in=state:{STATE_CODE}{county_str}'
    f'&key={CENSUS_KEY}'
)

print('Pulling ACS 2023 data for state tracts...')
response = requests.get(url)
data = response.json()
# Keeping only specified county data
state_tracts = pd.DataFrame(data[1:], columns=data[0])
state_tracts = state_tracts[state_tracts['county'].isin(COUNTIES)]
#Replace NaN equivalent number with NaN data
for col in CENSUS_VARIABLES:
    state_tracts[col] = pd.to_numeric(state_tracts[col], errors='coerce')
state_tracts.replace(-666666666, np.nan, inplace=True)
#Create unique tract ID
state_tracts['GEOID'] = state_tracts['state'] + state_tracts['county'] + state_tracts['tract']
print(f'Tracts loaded: {state_tracts.shape[0]}')
state_tracts.head()

Pulling ACS 2023 data for state tracts...
Tracts loaded: 3302


,NAME,B01003_001E,B08201_001E,B08201_002E,B08201_003E,B08301_001E,B08301_002E,B08301_004E,B08301_010E,B23025_001E,B23025_004E,B25003_001E,B25003_003E,B19013_001E,state,county,tract,GEOID
0,Census Tract 1011.10; Los Angeles County; Cali...,4152,1558,170,415,2109,1851,259,55,3556,2154,1558,760,84091.0,06,037,101110,06037101110
1,Census Tract 1011.22; Los Angeles County; Cali...,4198,1407,0,260,1991,1526,95,12,3652,2049,1407,311,99583.0,06,037,101122,06037101122
2,Census Tract 1012.20; Los Angeles County; Cali...,3434,1357,56,455,1622,1288,77,109,2791,1650,1357,761,69676.0,06,037,101220,06037101220
3,Census Tract 1012.21; Los Angeles County; Cali...,3931,1483,265,446,1805,1585,208,33,3228,1834,1483,1154,53798.0,06,037,101221,06037101221
4,Census Tract 1012.22; Los Angeles County; Cali...,2572,948,172,314,1131,900,52,59,2072,1198,948,873,45662.0,06,037,101222,06037101222


### Load TIGER tract boundaries and join demographics

In [ ]:
print('Downloading TIGER 2023 tract boundaries (CA)...')
tract_url = 'https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_' + STATE_CODE + '_tract.zip'

state_tract_geo = gpd.read_file(tract_url)
#Keep only tract boundaries within specified counties
state_tract_geo = state_tract_geo[state_tract_geo['COUNTYFP'].isin(COUNTIES)]
state_tract_geo = state_tract_geo.to_crs('EPSG:4326')
print(f'Tract geometries: {state_tract_geo.shape[0]}')
#Merge datasets with tract information and tract geometry
tracts = state_tract_geo[['GEOID', 'ALAND', 'geometry']].merge(
    state_tracts[['GEOID'] + CENSUS_VARIABLES],
    on='GEOID', how='left'
)
tracts.head()

Tract geometries: 3302


,GEOID,ALAND,geometry,B01003_001E,B08201_001E,B08201_002E,B08201_003E,B08301_001E,B08301_002E,B08301_004E,B08301_010E,B23025_001E,B23025_004E,B25003_001E,B25003_003E,B19013_001E
0,06037204920,909972,"POLYGON ((-118.20284 34.01966, -118.20283 34.0...",2459,674,79,190,947,829,153,39,1939,954,674,379,70500.0
1,06037205110,286962,"POLYGON ((-118.21964 34.02628, -118.21945 34.0...",3690,866,93,397,1401,1061,234,242,2686,1466,866,866,52262.0
2,06037320101,680504,"POLYGON ((-118.43882 34.29911, -118.43732 34.3...",3402,941,19,123,1724,1499,197,48,2750,1803,941,173,110967.0
3,06037205120,1466242,"POLYGON ((-118.22023 34.02056, -118.22018 34.0...",3409,935,218,373,1161,912,218,96,2513,1206,935,848,28516.0
4,06037206010,1418137,"POLYGON ((-118.23924 34.06726, -118.238 34.067...",3490,1497,392,654,1858,1271,256,140,3124,1919,1497,1389,60703.0


#### Engineer Model Features

In [ ]:
#Add engineered features to tracts
def engineer_features(df):
    out = df.copy()
    out['population_density']    = out['B01003_001E'] / (out['ALAND'] / 1_000_000)
    out['low_car_rate']          = (out['B08201_002E'] + out['B08201_003E']) / out['B08201_001E']
    out['transit_commuter_rate'] = out['B08301_010E'] / out['B08301_001E']
    out['driver_rate']           = out['B08301_002E'] / out['B08301_001E']
    out['carpool_rate']          = out['B08301_004E'] / out['B08301_001E']
    out['employment_rate']       = out['B23025_004E'] / out['B23025_001E']
    out['renter_rate']           = out['B25003_003E'] / out['B25003_001E']
    out['median_income']         = out['B19013_001E']
    return out

tracts = engineer_features(tracts)
tracts[CENSUS_FEATURE_COLS].describe()

,population_density,low_car_rate,transit_commuter_rate,driver_rate,carpool_rate,employment_rate,renter_rate,median_income
count,3297.000000,3269.000000,3273.000000,3273.000000,3273.000000,3279.000000,3269.000000,3253.000000
mean,4618.501558,0.387612,0.037145,0.762040,0.092532,0.603806,0.502209,99832.666462
std,4065.555692,0.175694,0.055117,0.123514,0.058888,0.086442,0.263903,41545.539502
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7000.000000
25%,2216.089570,0.255814,0.004397,0.697143,0.049970,0.558613,0.282380,70167.000000
50%,3670.064037,0.363746,0.018118,0.786753,0.084708,0.606734,0.491132,93264.000000
75%,5926.913638,0.494718,0.046419,0.850433,0.126606,0.653216,0.719384,120890.000000
max,46862.889282,1.000000,0.606532,1.000000,1.000000,1.000000,1.000000,250001.000000


In [56]:
def dist_to_point(lat, lon, clat, clon):
    R = 6371
    dlat, dlon = np.radians(lat - clat), np.radians(lon - clon)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat))*np.cos(np.radians(clat))*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def dist_to_nearest_job_center(lat, lon, centers):
    dists = [dist_to_point(lat, lon, c[0], c[1]) for c in centers]
    return min(dists)


### Load Job-center Data (LODES)

In [57]:
lodes_url = 'https://lehd.ces.census.gov/data/lodes/LODES8/' + STATE_ABBV + '/wac/' + STATE_ABBV + '_wac_S000_JT00_2021.csv.gz'

lodes = pd.read_csv(lodes_url, compression='gzip', dtype={'w_geocode': str})

# w_geocode is an 11-digit census block — first 11 chars = tract GEOID
lodes['GEOID'] = lodes['w_geocode'].str[:11]

# Aggregate jobs to tract level
jobs_by_tract = lodes.groupby('GEOID')['C000'].sum().reset_index()
jobs_by_tract.rename(columns={'C000': 'total_jobs'}, inplace=True)

print(jobs_by_tract.sort_values('total_jobs', ascending=False).head(10))

            GEOID  total_jobs
1977  06037207400      123890
6529  06073008511      106997
7061  06075011700       88928
2528  06037310400       85246
4600  06059075516       79766
4469  06059062610       78095
7980  06085505006       67760
7269  06075061501       61137
3848  06037980013       56240
2345  06037267901       54447


In [ ]:
#Merge job information into tract, calculate job density
tracts = tracts.merge(jobs_by_tract, on='GEOID', how='left')
tracts['job_density'] = tracts['total_jobs'] / (tracts['ALAND'] / 1_000_000)

#### Filter Job Centers

In [ ]:
#Filter out tracts with job density above 95th percentile to get true job centers
emp_threshold = np.percentile(tracts['job_density'].dropna(), 95)
job_dense_tracts = tracts[tracts['job_density'] >= emp_threshold]
print(f'Tracts above 95th percentile of job density: {len(job_dense_tracts)}')

filtered_centers = []
#Keeep job centers that are within a certain distance to avoid redundant job centers
for _, row in job_dense_tracts.sort_values('job_density', ascending=False).iterrows():
    lat = row.geometry.centroid.y
    lon = row.geometry.centroid.x
    
    if not filtered_centers:
        filtered_centers.append((lat, lon))
        continue
    
    # compute distance to all already accepted centers
    dists = [
        dist_to_point(lat, lon, c[0], c[1])
        for c in filtered_centers
    ]
    
    if min(dists) >= MIN_JOB_KM:
        filtered_centers.append((lat, lon))

job_centers = np.array(filtered_centers)

print(f'Filtered job centers: {len(job_centers)}')
print(job_centers[:10])


Tracts above 95th percentile of job density: 165
Filtered job centers: 66
[[  34.05670305 -118.44482529]
 [  34.05577403 -118.24559005]
 [  34.2026769  -118.33540664]
 [  34.0475522  -118.42019816]
 [  34.04635064 -118.26477756]
 [  34.07295691 -118.38008973]
 [  34.05186623 -118.3305985 ]
 [  34.0595607  -118.30222612]
 [  34.15353328 -118.33435693]
 [  34.09622773 -118.28840361]]


### Generate Candidate Grid

In [ ]:
def generate_grid(bounds, spacing_km):
    min_lon, min_lat, max_lon, max_lat = bounds
    deg_lat = spacing_km / 110.574
    deg_lon = spacing_km / (111.320 * np.cos(np.radians((min_lat + max_lat) / 2)))
    lats = np.arange(min_lat, max_lat, deg_lat)
    lons = np.arange(min_lon, max_lon, deg_lon)
    grid_lons, grid_lats = np.meshgrid(lons, lats)
    pts = pd.DataFrame({'lon': grid_lons.ravel(), 'lat': grid_lats.ravel()})
    return gpd.GeoDataFrame(
        pts,
        geometry=[Point(x, y) for x, y in zip(pts.lon, pts.lat)],
        crs='EPSG:4326'
    )

grid = generate_grid(BOUNDS, GRID_SPACING_KM)
print(f'Total grid points: {len(grid)}')

# Spatial join: each grid point inherits features of the tract it falls inside
grid_joined = gpd.sjoin(
    grid,
    tracts[['GEOID', 'ALAND', 'geometry'] + CENSUS_FEATURE_COLS],
    how='left',
    predicate='within'
)
# Drop points outside county boundary
grid_joined = grid_joined.dropna(subset=['GEOID']).reset_index(drop=True)
grid_joined['is_post_covid'] = 1
grid_joined['Week']  = WEEK if WEEK != '' else 24
grid_joined['Month'] = grid_joined['Week'].apply(lambda w: min(math.ceil(w / (52/12)), 12))
print(f'Candidates inside bounds: {len(grid_joined)}')
grid_joined['dist_to_nearest_job_center_km'] = grid_joined.apply(
    lambda r: dist_to_nearest_job_center(r.lat, r.lon, job_centers), axis=1
)

# Fill sparse NaNs with tract medians
for col in FEATURE_COLS:
    grid_joined[col] = grid_joined[col].fillna(grid_joined[col].median())

grid_joined[['lon','lat'] + FEATURE_COLS].head()

Total grid points: 64856
Candidates inside bounds: 39017


,lon,lat,Month,Week,population_density,low_car_rate,transit_commuter_rate,driver_rate,carpool_rate,employment_rate,renter_rate,median_income,is_post_covid,dist_to_nearest_job_center_km
0,-118.548251,33.355132,6,24,2.143161,0.320755,0.068311,0.296015,0.170778,0.48827,1.0,113929.0,1,53.671529
1,-118.542839,33.355132,6,24,2.143161,0.320755,0.068311,0.296015,0.170778,0.48827,1.0,113929.0,1,53.456480
2,-118.537428,33.355132,6,24,2.143161,0.320755,0.068311,0.296015,0.170778,0.48827,1.0,113929.0,1,53.245283
3,-118.532016,33.355132,6,24,2.143161,0.320755,0.068311,0.296015,0.170778,0.48827,1.0,113929.0,1,53.037984
4,-118.526605,33.355132,6,24,2.143161,0.320755,0.068311,0.296015,0.170778,0.48827,1.0,113929.0,1,52.834630


## Predict Using Model

In [61]:
model = xgb.XGBRegressor()
model.load_model(MODEL_PATH)

X = grid_joined[FEATURE_COLS].values
grid_joined['predicted_ridership'] = np.expm1(model.predict(X))

print(f"Score range: {grid_joined['predicted_ridership'].min():.0f} – {grid_joined['predicted_ridership'].max():.0f}")
grid_joined[['lon','lat','predicted_ridership']].sort_values('predicted_ridership', ascending=False).head(10)


Score range: 4400 – 48540


,lon,lat,predicted_ridership
14133,-118.277673,34.024367,48539.671875
14377,-118.277673,34.028889,47071.542969
14376,-118.283085,34.028889,47071.542969
14132,-118.283085,34.024367,45787.039062
15132,-118.256027,34.042455,43342.179688
15901,-118.283085,34.056020,40556.789062
15134,-118.245204,34.042455,40197.250000
15647,-118.250616,34.051498,39775.441406
14627,-118.261439,34.033411,38743.164062
14877,-118.266850,34.037933,38743.164062


## Greedy Station Selection

In [ ]:
#Calculate distance between points
def haversine(lat1, lon1, lat2_arr, lon2_arr):
    R = 6371.0
    dlat = np.radians(lat2_arr - lat1)
    dlon = np.radians(lon2_arr - lon1)
    a = (np.sin(dlat/2)**2
         + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2_arr)) * np.sin(dlon/2)**2)
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
#Select candidates within generated grid that have highest ridership
def greedy_select(candidates, n, min_spacing_km):
    ranked = candidates.sort_values('predicted_ridership', ascending=False)
    selected = []
    sel_lats, sel_lons = [], []

    for idx, row in ranked.iterrows():
        if len(selected) >= n:
            break
        if sel_lats:
            dists = haversine(row['lat'], row['lon'],
                              np.array(sel_lats), np.array(sel_lons))
            if dists.min() < min_spacing_km:
                continue
        selected.append(idx)
        sel_lats.append(row['lat'])
        sel_lons.append(row['lon'])

    return selected

selected_idx = greedy_select(grid_joined, N_STATIONS, MIN_SPACING_KM)
selected = grid_joined.loc[selected_idx].reset_index(drop=True)

print(f'Selected {len(selected)} stations')
print(f'Total predicted ridership: {selected["predicted_ridership"].sum():.0f}')
selected[['lat','lon','predicted_ridership'] + CENSUS_FEATURE_COLS].round(4)

Selected 25 stations
Total predicted ridership: 612403


,lat,lon,predicted_ridership,population_density,low_car_rate,transit_commuter_rate,driver_rate,carpool_rate,employment_rate,renter_rate,median_income
0,34.0244,-118.2777,48539.671875,6340.2820,0.9276,0.0341,0.2652,0.0404,0.3251,1.0000,11270.0
1,34.0560,-118.2831,40556.789062,17104.9947,0.9224,0.5451,0.3005,0.0239,0.4831,0.9955,32361.0
2,34.0425,-118.2452,40197.250000,41015.0360,1.0000,0.3376,0.2325,0.1624,0.1770,1.0000,7000.0
3,34.1012,-118.3426,35143.023438,7126.6212,0.8992,0.1751,0.5119,0.0429,0.5807,1.0000,42458.0
4,34.0877,-118.2885,31690.923828,10800.1349,0.7344,0.3093,0.5074,0.0372,0.6369,0.9134,53699.0
5,34.1962,-118.4887,28425.076172,21.6671,1.0000,0.0000,1.0000,0.0000,0.4706,1.0000,113929.0
6,33.9837,-118.3318,24818.259766,10129.1438,0.6923,0.2197,0.7378,0.0143,0.4770,0.8647,28415.0
7,34.0425,-118.3156,23985.080078,7051.8993,0.6383,0.2476,0.5956,0.0408,0.5871,0.8826,37285.0
8,34.0605,-118.4454,23187.541016,5526.3198,0.7510,0.0000,0.2966,0.0000,0.5161,0.7995,117829.0
9,33.6491,-117.8447,21385.910156,3572.4949,0.5724,0.0425,0.3059,0.0502,0.4866,0.6811,80167.0


### Route Generation (not necessary just for fun)

In [63]:
from scipy.optimize import linprog
import itertools
PERP_KM = 1.5

In [64]:
def straightness_score(lats, lons):
    if len(lats) < 2:
        return 0
    if len(np.unique(lons)) == 1:  # perfectly vertical line
        return 1.0
    if len(np.unique(lats)) == 1:  # perfectly horizontal line
        return 1.0
    # Try both orientations and take the best
    r2_a = LinearRegression().fit(lons.reshape(-1,1), lats).score(lons.reshape(-1,1), lats)
    r2_b = LinearRegression().fit(lats.reshape(-1,1), lons).score(lats.reshape(-1,1), lons)
    return max(r2_a, r2_b)

In [65]:
def generate_all_candidate_lines(selected, min_stations=3, perp_threshold_km=PERP_KM):
    """
    Generate every possible straight-ish line through subsets of stations.
    Returns list of (station_indices, score) tuples.
    """
    candidates = []
    n = len(selected)
    coords = selected[['lat', 'lon']].values
    ridership = selected['predicted_ridership'].values
    
    # For each pair of stations, define a line direction
    # then find all stations close to that line
    for i, j in itertools.combinations(range(n), 2):
        # Direction vector between stations i and j
        direction = coords[j] - coords[i]
        norm = np.linalg.norm(direction)
        if norm < 1e-9:
            continue
        direction = direction / norm
        origin = coords[i]
        lat_rad = np.radians(origin[0])
        
        # Find all stations within perp_threshold of this line
        on_line = []
        for k in range(n):
            v = coords[k] - origin
            proj = np.dot(v, direction) * direction
            perp = v - proj
            perp_km = np.sqrt(
                (perp[0] * 110.574)**2 +
                (perp[1] * 111.320 * np.cos(lat_rad))**2
            )
            if perp_km < perp_threshold_km:
                on_line.append(k)
        
        if len(on_line) < min_stations:
            continue
        
        # Score this candidate line
        subset_lats = coords[on_line, 0]
        subset_lons = coords[on_line, 1]
        r2 = straightness_score(subset_lats, subset_lons)
        total_ridership = ridership[on_line].sum()
        score = total_ridership * (r2 ** 2)  # square R² to penalize jaggedness harder
        
        candidates.append((on_line, score))
    
    # Deduplicate — same set of stations from different anchor pairs
    seen = set()
    unique = []
    for station_indices, score in candidates:
        key = frozenset(station_indices)
        if key not in seen:
            seen.add(key)
            unique.append((station_indices, score))
    
    return sorted(unique, key=lambda x: x[1], reverse=True)

def jaccard_similarity(set1, set2):
    intersection = len(set(set1) & set(set2))
    union = len(set(set1) | set(set2))
    return intersection / union

def select_lines_greedy(candidates, selected, max_lines=MAX_LINES,
                        interchange_percentile=60, max_similarity=0.7):
    """
    Greedily pick the best non-redundant lines.
    A line is redundant if >max_similarity% of its stations are already covered.
    High ridership stations can appear on multiple lines (interchanges).
    """
    interchange_threshold = np.percentile(
        selected['predicted_ridership'], interchange_percentile
    )
    
    covered = set()
    line_orders = {}
    selected_line_sets = []  # track station sets of chosen lines
    line_id = 0
    
    for station_indices, score in candidates:
        if line_id >= max_lines:
            break
        
        too_similar = any(
            jaccard_similarity(station_indices, existing) > max_similarity
            for existing in selected_line_sets
        )
        if too_similar:
            continue
        
        # Skip if too many non-interchange stations already covered
        non_interchange_covered = [
            i for i in station_indices
            if i in covered and 
            selected.iloc[i]['predicted_ridership'] < interchange_threshold
        ]
        if len(non_interchange_covered) / len(station_indices) > 0.5:
            continue
        
         # Order stations along the line
        coords = selected[['lat', 'lon']].values
        subset_coords = coords[station_indices]
        centered = subset_coords - subset_coords.mean(axis=0)
        _, _, Vt = np.linalg.svd(centered)
        direction = Vt[0]
        projections = centered @ direction
        order = np.argsort(projections)
        ordered_indices = [station_indices[i] for i in order]
        df_indices = selected.iloc[ordered_indices].index.tolist()
        
        line_orders[line_id] = df_indices
        selected_line_sets.append(set(station_indices))
        
        # Mark non-interchange stations as covered
        for i in station_indices:
            if selected.iloc[i]['predicted_ridership'] < interchange_threshold:
                covered.add(i)
        
        r2 = straightness_score(
            selected.iloc[ordered_indices]['lat'].values,
            selected.iloc[ordered_indices]['lon'].values
        )
        ridership = selected.iloc[ordered_indices]['predicted_ridership'].sum()
        print(f'Line {line_id+1}: {len(ordered_indices)} stations | '
              f'R²={r2:.3f} | ridership={ridership:.0f} | score={score:.0f}')
        
        line_id += 1
    
    return line_orders

In [66]:
print('Generating candidate lines...')
candidates = generate_all_candidate_lines(
    selected, 
    min_stations=3, 
    perp_threshold_km=PERP_KM
)
print(f'Unique candidate lines: {len(candidates)}')

print('\nSelecting best lines...')
line_orders = select_lines_greedy(
    candidates, selected, MAX_LINES
)

print(f'\nTotal lines selected: {len(line_orders)}')

for line_id, station_idx in line_orders.items():
    print(f'\nLine {line_id + 1}:')
    print(selected.loc[station_idx][['lat', 'lon', 'predicted_ridership']].to_string())

Generating candidate lines...
Unique candidate lines: 141

Selecting best lines...
Line 1: 7 stations | R²=0.987 | ridership=213595 | score=207924
Line 2: 7 stations | R²=0.997 | ridership=196674 | score=195319
Line 3: 6 stations | R²=0.999 | ridership=182863 | score=182374
Line 4: 5 stations | R²=0.996 | ridership=165921 | score=164702
Line 5: 6 stations | R²=0.992 | ridership=159957 | score=157292
Line 6: 5 stations | R²=0.986 | ridership=153323 | score=149034
Line 7: 5 stations | R²=0.967 | ridership=151789 | score=141865
Line 8: 4 stations | R²=0.995 | ridership=141117 | score=139664
Line 9: 6 stations | R²=0.929 | ridership=158636 | score=137013
Line 10: 4 stations | R²=0.979 | ridership=134579 | score=128876

Total lines selected: 10

Line 1:
          lat         lon  predicted_ridership
22  34.024367 -118.212735         18681.605469
2   34.042455 -118.245204         40197.250000
1   34.056020 -118.283085         40556.789062
4   34.087673 -118.288497         31690.925781
3   34

## MAPPING

In [67]:
cmap = plt.cm.YlOrRd
vmin = selected['predicted_ridership'].min()
vmax = selected['predicted_ridership'].max()

def score_to_hex(val):
    norm = (val - vmin) / (vmax - vmin + 1e-9)
    r, g, b, _ = cmap(norm)
    return '#{:02x}{:02x}{:02x}'.format(int(r*255), int(g*255), int(b*255))

m = folium.Map(location=[BOUND_AVG[1], BOUND_AVG[0]], zoom_start=11, tiles='CartoDB positron')

# Background heatmap of all candidate scores
heat_data = grid_joined[['lat','lon','predicted_ridership']].values.tolist()
HeatMap(heat_data, radius=14, blur=18, min_opacity=0.3,
        gradient={'0.3':'blue','0.55':'lime','0.75':'orange','1.0':'red'}).add_to(m)

# Selected station markers
for i, row in selected.iterrows():
    color = score_to_hex(row['predicted_ridership'])
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=10,
        color='white', weight=2,
        fill=True, fill_color=color, fill_opacity=0.95,
        tooltip=(
            f"<b>Station {i+1}</b><br>"
            f"Predicted ridership: <b>{row['predicted_ridership']:.0f}</b><br>"
            f"Pop density: {row['population_density']:.0f}/km²<br>"
            f"Transit rate: {row['transit_commuter_rate']:.1%}<br>"
            f"Low-car rate: {row['low_car_rate']:.1%}<br>"
            f"Employment rate: {row['employment_rate']:.1%}<br>"
            f"Renter rate: {row['renter_rate']:.1%}<br>"
            f"Median income: ${row['median_income']:,.0f}"
        )
    ).add_to(m)

legend_html = """
<div style='position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:12px 16px;border-radius:8px;
     box-shadow:2px 2px 8px rgba(0,0,0,.3);font-family:sans-serif;font-size:13px'>
  <b>Predicted Ridership</b><br>
  <span style='color:#d73027'>●</span> High &nbsp;
  <span style='color:#fee08b'>●</span> Medium &nbsp;
  <span style='color:#4575b4'>●</span> Low<br><br>
  <i>Hover markers for tract features</i>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m.save('transit_stations.html')
print('Saved → transit_stations.html')
m

Saved → transit_stations.html


### Mapping with Routes Generated (not necessary just for fun)

In [68]:
# LINE_COLORS = [
#     '#e41a1c','#377eb8','#4daf4a','#ff7f00',
#     '#984ea3','#a65628','#f781bf','#999999',
#     '#00ced1','#ffd700'
# ]

# cmap = plt.cm.YlOrRd
# vmin = selected['predicted_ridership'].min()
# vmax = selected['predicted_ridership'].max()

# def score_to_hex(val):
#     norm = (val - vmin) / (vmax - vmin + 1e-9)
#     r, g, b, _ = cmap(norm)
#     return '#{:02x}{:02x}{:02x}'.format(int(r*255), int(g*255), int(b*255))

# m = folium.Map(location=[BOUND_AVG[1], BOUND_AVG[0]], zoom_start=11, tiles='CartoDB positron')

# # Background heatmap
# heat_data = grid_joined[['lat','lon','predicted_ridership']].values.tolist()
# HeatMap(heat_data, radius=14, blur=18, min_opacity=0.3,
#         gradient={'0.3':'blue','0.55':'lime','0.75':'orange','1.0':'red'}).add_to(m)

# # # Draw lines first
# # for line_id, station_idx in line_orders.items():
# #     color = LINE_COLORS[line_id % len(LINE_COLORS)]
# #     line_stations = selected.loc[station_idx]
# #     coords = [[row['lat'], row['lon']] for _, row in line_stations.iterrows()]
# #     folium.PolyLine(
# #         coords, color=color, weight=5, opacity=0.8,
# #         tooltip=f'Line {line_id + 1}'
# #     ).add_to(m)

# # All selected stations — grey if not on any line
# assigned_idx = set(idx for idxs in line_orders.values() for idx in idxs)

# for i, row in selected.iterrows():
#     if i not in assigned_idx:
#         folium.CircleMarker(
#             location=[row['lat'], row['lon']],
#             radius=7,
#             color='white', weight=2,
#             fill=True, fill_color='#888888', fill_opacity=0.7,
#             tooltip=(
#                 f"<b>Unassigned Station</b><br>"
#                 f"Predicted ridership: <b>{row['predicted_ridership']:.0f}</b><br>"
#                 f"Pop density: {row['population_density']:.0f}/km²<br>"
#                 f"Transit rate: {row['transit_commuter_rate']:.1%}<br>"
#                 f"Low-car rate: {row['low_car_rate']:.1%}<br>"
#                 f"Employment rate: {row['employment_rate']:.1%}<br>"
#                 f"Renter rate: {row['renter_rate']:.1%}<br>"
#                 f"Median income: ${row['median_income']:,.0f}<br>"
#                 f"Job proximity: {row['dist_to_nearest_job_center_km']:.1f} km"
#             )
#         ).add_to(m)

# # Station markers colored by line
# for line_id, station_idx in line_orders.items():
#     color = LINE_COLORS[line_id % len(LINE_COLORS)]
#     line_stations = selected.loc[station_idx]
#     for i, (idx, row) in enumerate(line_stations.iterrows()):
#         is_terminal = (i == 0 or i == len(line_stations) - 1)
#         is_interchange = selected.loc[idx, 'is_interchange'] if 'is_interchange' in selected.columns else False
#         folium.CircleMarker(
#             location=[row['lat'], row['lon']],
#             radius=12 if is_interchange else (10 if is_terminal else 7),
#             color='white', weight=2,
#             fill=True, fill_color=color, fill_opacity=0.95,
#             tooltip=(
#                 f"<b>Line {line_id+1} — Station {i+1}</b><br>"
#                 f"{'🔀 INTERCHANGE — ' if is_interchange else ''}"
#                 f"{'⬤ TERMINAL — ' if is_terminal else ''}"
#                 f"Predicted ridership: <b>{row['predicted_ridership']:.0f}</b><br>"
#                 f"Pop density: {row['population_density']:.0f}/km²<br>"
#                 f"Transit rate: {row['transit_commuter_rate']:.1%}<br>"
#                 f"Low-car rate: {row['low_car_rate']:.1%}<br>"
#                 f"Employment rate: {row['employment_rate']:.1%}<br>"
#                 f"Renter rate: {row['renter_rate']:.1%}<br>"
#                 f"Median income: ${row['median_income']:,.0f}<br>"
#                 f"Job proximity: {row['dist_to_nearest_job_center_km']:.1f} km"
#             )
#         ).add_to(m)

# # Legend
# legend_items = ''.join([
#     f"<span style='color:{LINE_COLORS[i % len(LINE_COLORS)]}'>━━</span> Line {i+1}&nbsp;&nbsp;"
#     for i in range(len(line_orders))
# ])
# legend_html = f"""
# <div style='position:fixed;bottom:30px;left:30px;z-index:1000;
#      background:white;padding:12px 16px;border-radius:8px;
#      box-shadow:2px 2px 8px rgba(0,0,0,.3);font-family:sans-serif;font-size:13px;max-width:400px'>
#   <b>Transit Lines</b><br>{legend_items}<br><br>
#   <span style='font-size:11px'>
#   ⬤ Terminal &nbsp;|&nbsp; 🔀 Interchange &nbsp;|&nbsp; ⬤ Unassigned &nbsp;|&nbsp; Hover for details
#   </span>
# </div>
# """
# m.get_root().html.add_child(folium.Element(legend_html))

# m.save('transit_stations.html')
# print('Saved → transit_stations.html')
# m